# Comprehensive Weighted Out-of-Range Recovery Study: Hold-400 Cosine Decay

This notebook compares four recovery prompts across four S10000 Christoffel sampling laws, uniform MCS, and pure inverse-square sampling at $m/n\in\{0.01,0.02,0.03,0.04,0.05\}$. Every cell uses five trials, CFG scale 1, learning rate 0.1 through iteration 400 followed by cosine decay to 0.001 at iteration 2,000, weighted least squares, and a unitary Fourier operator. The Christoffel laws use $\zeta=1/2$; the strictly positive baselines use their original laws.

In [ ]:
from pathlib import Path
import importlib.util
from IPython.display import display

cwd = Path.cwd().resolve()
study_relpath = Path('analyze_results/weighted/out_of_range')
candidates = [cwd, cwd / study_relpath]
candidates.extend(parent / study_relpath for parent in cwd.parents)
HELPER_ROOT = next((path for path in candidates if (path / 'analysis.py').is_file()), None)
if HELPER_ROOT is None:
    raise FileNotFoundError(f'Could not locate {study_relpath}/analysis.py.')
module_path = HELPER_ROOT / 'analysis.py'
module_spec = importlib.util.spec_from_file_location('weighted_out_of_range_analysis', module_path)
diagnostic = importlib.util.module_from_spec(module_spec)
module_spec.loader.exec_module(diagnostic)
recovery = diagnostic.recovery
ROOT = diagnostic.PROJECT_ROOT
RUN_DIR = diagnostic.RUN_ROOT
OUTPUT_DIR = RUN_DIR / 'figures'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
# Give the large paper-style labels enough room in this compact diagnostic.
recovery.SWEEP_SINGLE_FIGSIZE_HEIGHT = 6.2
recovery.SWEEP_SINGLE_LEGEND_Y = 0.9
recovery.SWEEP_SINGLE_BOTTOM = 0.24
recovery.SWEEP_SINGLE_TOP = 0.70
recovery.SWEEP_SINGLE_XLABEL_Y = 0.055
def refresh_rows():
    recovery.ensure_lpips_metrics(
        ROOT,
        result_namespace='weighted',
        artifact_roots=[diagnostic.RESULT_ROOT],
        device='cpu',
        metrics_path=diagnostic.RESULT_ROOT / 'lpips_metrics.csv',
    )
    return diagnostic.load_rows(include_partial=True)
print('Expected reconstructions:', diagnostic.EXPECTED_ROWS)

## Optimization trajectories and trial uncertainty

One 2-by-3 objective figure is exported for each sampling law. The first five panels show $m/N=0.01,\ldots,0.05$, and the final panel contains the legend. Thin curves show individual trials and bold curves show their arithmetic mean. Confidence intervals remain available through `show_confidence_interval=True`; they are disabled below so invisible bounds do not affect the y-axis scale. Publication figures are saved only as PDFs and rendered inline as SVG output for copying.

In [ ]:
TRACES = diagnostic.load_optimization_traces()
TRACE_SUMMARY = diagnostic.optimization_trace_summary(TRACES)
TRACE_OUTPUTS = diagnostic.export_optimization_trace_figures(
    TRACES,
    OUTPUT_DIR,
    metrics=('bp_loss',),
    show_confidence_interval=False,
    show=True,
)
print(f'Loaded {len(TRACES):,} saved trial-iteration rows.')
TRACE_OUTPUTS

## Completion and compatibility audit

In [ ]:
ROWS = refresh_rows()
COMPLETION = diagnostic.completion_table(ROWS)
COMPLETION.to_csv(OUTPUT_DIR / 'completion.csv', index=False)
display(COMPLETION.groupby(['sampling_condition', 'reconstruction_condition'], as_index=False)[['observed', 'expected', 'left']].sum())
print(f'Observed {int(COMPLETION.observed.sum())} / {diagnostic.EXPECTED_ROWS} reconstructions.')

## Sampling-ratio sweeps

In [ ]:
ROWS = refresh_rows()
METRIC_OUTPUTS = recovery.export_metric_figures(
    ROWS,
    OUTPUT_DIR,
    sweep_metrics=('psnr_db', 'ssim', 'lpips', 'pixel_mae'), #, 'runtime_sec'),
    combined_metrics=(),
    combine_sampling_methods=True,
    show=True,
)
METRIC_OUTPUTS

## Reconstruction panels

In [ ]:
ROWS = refresh_rows()
all = [0.01, 0.02, 0.03, 0.04, 0.05]
GRID_OUTPUTS = {}
for sampling_ratio in all: #[0.03]: #all:
    GRID_OUTPUTS[sampling_ratio] = recovery.export_recovery_grids(
        ROWS,
        ROOT,
        OUTPUT_DIR,
        sampling_method=None,
        sampling_percentage=sampling_ratio,
        show=False,
    )
GRID_OUTPUTS

## Runtime summary

In [ ]:
if ROWS.empty:
    print('No completed reconstructions yet.')
else:
    RUNTIME_SUMMARY = (
        ROWS.groupby(['sampling_condition', 'reconstruction_condition', 'samp_perc'], as_index=False)
        .agg(runtime_mean_sec=('runtime_sec', 'mean'), runtime_std_sec=('runtime_sec', 'std'), trials=('runtime_sec', 'count'))
    )
    RUNTIME_SUMMARY.to_csv(OUTPUT_DIR / 'runtime_summary.csv', index=False)
    display(RUNTIME_SUMMARY)